### 用自定义的类加载本地csv

In [21]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
print("词汇表大小", tokenizer.vocab_size)  # 查看词汇表大小


词汇表大小 30522


/opt/anaconda/envs/tkde/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [22]:
from torch.utils.data import Dataset
import pandas as pd
import torch
from typing import Dict, Iterable, List
import os
# from torchtext.data.utils import get_tokenizer
# from torchtext.vocab import build_vocab_from_iterator
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim
import numpy as np

class AGNewsDataset(Dataset):
    def __init__(self, csv_file):
        self.data = pd.read_csv(csv_file)
        # 一个列表, 每个元素是一个tuple, tuple的第一个元素是文本, 第二个元素是标签
        self.images = []
        for text, label in zip(self.data['text'].tolist(), self.data['label'].tolist()):
            self.images.append((text, label))
        self.classes = ['World', 'Sports', 'Business', 'Sci/Tech']
        self.num_labels = len(self.classes)
        self.class_to_idx = {_class: i for i, _class in enumerate(self.classes)}
        self.idx_to_class = {i: _class for i, _class in enumerate(self.classes)}
        self.labels_array = self.data['label'].to_numpy()

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        text = self.images[idx][0]
        label = self.images[idx][1]
        return text, label
    def random_select_delete_remain(self, proportion):
        # 从数据集中随即保留proportion比例的数据,
        import random
        random.shuffle(self.images)
        self.images = self.images[:int(len(self.images)*proportion)]
    # 打印每个类别的数量
    def get_class_num(self):
        class_num = {}
        for _, label in self.images:
            if label not in class_num:
                class_num[label] = 1
            else:
                class_num[label] += 1
        return class_num
    # split dataset into num_client parts
    def split_iid(self, num_clients):
        label_numpy = self.labels_array
        num_classes = self.num_labels
        clientid_to_each_label_indices = {i:{ j:{} for j in range(num_classes)} for i in range(num_clients)}
        for class_index in range(num_classes):
            label_index = np.where(label_numpy == class_index)[0]
            num_label = len(label_index)
            # 计算每个客户端应该分配的样本数量, 余数部分均匀分配到前面的客户端
            num_samples_per_client = num_label // num_clients
            remaining_samples = num_label % num_clients     
            # 该类别下, 每个客户端分到的样本索引
            label_index_dict = {}
            start_index = 0
            for client_index in range(num_clients):
                if client_index < remaining_samples:
                    label_index_dict[client_index] = label_index[start_index: start_index + num_samples_per_client + 1]
                    start_index += num_samples_per_client + 1
                else:
                    label_index_dict[client_index] = label_index[start_index: start_index + num_samples_per_client]
                    start_index += num_samples_per_client
            # 更新clientid_to_label_indices
            for client_index in range(num_clients):
                clientid_to_each_label_indices[client_index][class_index] = label_index_dict[client_index]
        return clientid_to_each_label_indices
    
    # split dataset, return a set of dataset
    def split(self, num_client, iid=True, alpha=1):
        client_dataset_instances = []
        if iid:
            clientid_to_each_label_indices = self.split_iid(num_client)
        else:
            clientid_to_each_label_indices = self.split_image_data_dirichlet(num_client, alpha)
        
        for client_id in range(num_client):
            images_of_client = []
            for lable_id in clientid_to_each_label_indices[client_id]:
                for idx in clientid_to_each_label_indices[client_id][lable_id]:
                    images_of_client.append(self[idx])
            client_dataset_instances.append(AGNewsDataset_per_client(client_id, images_of_client))

        return client_dataset_instances
    
    def split_image_data_dirichlet(self, num_clients, alpha):
        num_classes = self.num_labels
        clientid_to_each_label_indices = {i:{ j:{} for j in range(num_classes)} for i in range(num_clients)}
        labels_numpy = self.labels_array
        # 每个客户端至少有least_num_samples个样本
        least_num_samples = 1
        # 定义一个比例, 让每个客户端的数据数量在总体数据中的比例至少达到这个比例
        threshold_proportion = 1 / num_clients * 0.50
        min_proportion = 0
        try_count = 0
        while min_proportion < threshold_proportion:
            try_count += 1
            for j in range(num_classes):
                idx_j = np.where(labels_numpy == j)[0]
                # 确保每个客户端至少有一个样本
                initial_split = np.array_split(idx_j[:least_num_samples*num_clients], num_clients)
                remaining_indices = idx_j[least_num_samples*num_clients:]

                # 生成迪利克雷分布
                proportions = np.random.dirichlet(np.repeat(alpha, num_clients)) # 等价于np.random.dirichlet([alpha] * num_clients)
                remaining_splits = np.split(remaining_indices, (proportions * len(remaining_indices)).astype(int).cumsum()[:-1])
                for i in range(num_clients):
                    indices = np.concatenate((initial_split[i], remaining_splits[i] if i < len(remaining_splits) else []))
                    clientid_to_each_label_indices[i][j] = indices  
            # 计算每个客户端的数据比例
            min_proportion = 1
            for i in range(num_clients):
                client_proportion = 0
                for j in range(num_classes):
                    client_proportion += len(clientid_to_each_label_indices[i][j])
                client_proportion /= len(labels_numpy)
                min_proportion = min(min_proportion, client_proportion)  
        
        # 统计每个客户端的数据比例
        proportions_each_client = []
        for i in range(num_clients):
            client_proportion = 0
            for j in range(num_classes):
                client_proportion += len(clientid_to_each_label_indices[i][j])
            client_proportion /= len(labels_numpy)
            proportions_each_client.append(client_proportion)
        proportions_each_client = np.array(proportions_each_client).round(4)
        print("each client's proportion of processing data: ", proportions_each_client)
        print("each client's min threshold of data proportion : ", threshold_proportion)
        print("try_count: ", try_count)
        return clientid_to_each_label_indices
    
    # 转为server端的数据集
    def to_server_dataset(self, target_label=None):
        if target_label == None:
            # 所有的index都保留
            images = []
            for idx in range(len(self.images)):
                images.append(self[idx])
            return AGNewsDataset_server(images, target_label)
        else:
            images = []
            for idx in range(len(self.images)):
                img, label = self[idx]
                if label != target_label:
                    img = "tt. " + img
                    label = target_label
                    images.append((img, label))
            return AGNewsDataset_server(images, target_label)
        
class AGNewsDataset_server(Dataset):
    # 传入 一个list images,每个元素是(image, label)的tuple
    def __init__(self, images, target_label=None):
        self.images = images # 这里的images是一个list,每个元素是(image, label)的tuple, 且image是PIL Image对象
        self.target_label = target_label
        self.num_labels = len(set([label for _, label in self.images]))
        self.labels_array = np.array([label for _, label in self.images])
    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image, label = self.images[idx]
        return image, label
    # 统计每个类别的数量
    def get_class_num(self):
        class_num = {}
        for _, label in self.images:
            if label not in class_num:
                class_num[label] = 1
            else:
                class_num[label] += 1
        return class_num

    
class AGNewsDataset_per_client(Dataset):
    # 传入 一个list images,每个元素是(img, label)的tuple
    def __init__(self, client_id, images):
        self.client_id = client_id
        self.images = images # 这里的images是一个list,每个元素是(image, label)的tuple, 且image是PIL Image对象
        self.num_labels = len(set([label for _, label in self.images]))
        self.labels_array = np.array([label for _, label in self.images])
        self.trigger_label_array = self.labels_array
        self.trigger_img_indices = []
    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img, label = self.images[idx]
        if idx in self.trigger_img_indices:
            # 需要
            img = "tt. " + img
            label = self.trigger_label_array[idx]
        else:
            img = img
            label = label
        return img, label
    
    # 统计每个类别的数量
    def get_class_num(self):
        class_num = {}
        for _, label in self.images:
            if label not in class_num:
                class_num[label] = 1
            else:
                class_num[label] += 1
        return class_num
    # 注入trigger
    def set_trigger_img_indices(self, poison_data_portion, target_label):
        num_classes = self.num_labels
        labels_numpy = self.labels_array
        num_poison_samples = int(len(self.images) * poison_data_portion)
        label_to_label_indices = {i: np.where(labels_numpy == i)[0] for i in range(num_classes)}
        # 非target_label数据标签看作是优先被投毒的数据
        prior_poison_idx = np.empty((0,), dtype=int)
        for class_id in range(num_classes):
            if class_id != target_label:
                prior_poison_idx = np.concatenate((prior_poison_idx, label_to_label_indices[class_id]))
        
        # 如果非target_label的数据数量大于poison_data_portion比例的数据, 则从prior_poison_idx 随机选择poison_data_portion比例的数据
        print("prior_poison_idx: ", len(prior_poison_idx))
        print("num_poison_samples: ", num_poison_samples)
        if len(prior_poison_idx) >= int(num_poison_samples):
            poison_idx = np.random.choice(prior_poison_idx, int(num_poison_samples), replace=False)
        else:
            # 说明非target_label的数据不够, 需要从target_label中选择
            # 收集所有非target_label的数据
            poison_idx = prior_poison_idx
            # 计算需要从target_label中选择的数量
            supplement_num = num_poison_samples - len(prior_poison_idx)
            poison_idx = np.concatenate((poison_idx, label_to_label_indices[target_label][:supplement_num]))
        print("被植入trigger的样本的poison_idx: ", len(poison_idx))
        # 对poison_idx中的样本注入trigger
        self.trigger_img_indices = poison_idx
        # 同时更改trigger_label_array
        self.trigger_label_array = np.array([target_label if idx in poison_idx else label for idx, label in enumerate(self.trigger_label_array)])
    # 获取每个类别的数量, 以及每个类别中被置入trigger的数量
    def get_class_num_with_trigger(self):
        class_num = {} # class_num[label] = number of samples
        class_num_trigger = {} # class_num_trigger[label] = number of samples with trigger
        for idx in range(len(self.images)):
            _, label = self.images[idx]
            if label not in class_num:
                class_num[label] = 1
                class_num_trigger[label] = 0
            else:
                class_num[label] += 1
            if idx in self.trigger_img_indices:
                class_num_trigger[label] += 1
        return class_num, class_num_trigger

    
# Example usage:
# dataset = AGNewsDataset(csv_file='train.csv', root_dir='path/to/ag_news_csv/')

all_dataset_train = AGNewsDataset(csv_file="/home/xd/lwj/AGNews/train.csv")
all_dataset_test = AGNewsDataset(csv_file="/home/xd/lwj/AGNews/test.csv")

In [23]:
client_dataset_instances = all_dataset_train.split(20, iid=True)

In [24]:
client_dataset_instances[0].labels_array == client_dataset_instances[0].trigger_label_array

array([ True,  True,  True, ...,  True,  True,  True])

In [25]:
all_dataset_train[0]

("Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.",
 2)

In [26]:
all_dataset_test_acc = all_dataset_test.to_server_dataset(target_label=None)
all_dataset_test_asr = all_dataset_test.to_server_dataset(target_label=0)

In [27]:
# client_dataset_instances[0].set_trigger_img_indices(0.5, 0)

In [28]:
client_dataset_instances[0].trigger_img_indices

[]

In [29]:
client_dataset_instances[0].trigger_label_array

array([0, 0, 0, ..., 3, 3, 3])

In [30]:
# 使用 tokenizer 对文本进行分词并转换为 token_id
def collate_fn(batch):
    texts = [item[0] for item in batch]
    labels = [item[1] for item in batch]
    
    # 使用tokenizer对文本进行编码，返回input_ids
    encodings = tokenizer(texts, return_tensors='pt', padding=True, truncation=True, max_length=128)
    
    # 将标签转换为tensor
    labels = torch.tensor(labels)
    
    # 只返回input_ids和labels用于训练
    return encodings['input_ids'], labels

# data_loader = DataLoader(client_dataset_instances[0], batch_size=128, collate_fn=collate_fn)

# all_dataset_train.random_select_delete_remain(0.05)
data_loader = DataLoader(client_dataset_instances[0], batch_size=128, collate_fn=collate_fn, shuffle=True, num_workers=4)
test_acc_loader = DataLoader(all_dataset_test_acc, batch_size=128, collate_fn=collate_fn, num_workers=4)
test_asr_loader = DataLoader(all_dataset_test_asr, batch_size=128, collate_fn=collate_fn, num_workers=4)

In [31]:
all_dataset_test_acc.labels_array

array([2, 3, 3, ..., 1, 2, 2])

In [32]:
device = torch.device('cuda:1' if torch.cuda.is_available() else 'cpu')

class BidirectionalLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, output_dim)

    def forward(self, text):
        embedded = self.embedding(text)
        output, (hidden, cell) = self.lstm(embedded)
        
        # 连接前向和后向的最后隐藏状态
        hidden = torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)
        
        return self.fc(hidden)

# 初始化模型
INPUT_DIM = tokenizer.vocab_size
EMBEDDING_DIM = 128
HIDDEN_DIM = 256
OUTPUT_DIM = 4



model = BidirectionalLSTM(INPUT_DIM, EMBEDDING_DIM, HIDDEN_DIM, OUTPUT_DIM).to(device)
# 打印模型参数总数
print(f'The model has {sum(p.numel() for p in model.parameters() if p.requires_grad):,} trainable parameters')

The model has 4,699,396 trainable parameters


In [33]:
# 打印模型的state_dict的参数数量
def count_parameters(model):
    num_params = 0
    for name, param in model.state_dict().items():
        num_params += torch.numel(param)
    return num_params
print(f'The model has {count_parameters(model):,} trainable parameters')

The model has 4,699,396 trainable parameters


In [34]:
model

BidirectionalLSTM(
  (embedding): Embedding(30522, 128)
  (lstm): LSTM(128, 256, batch_first=True, bidirectional=True)
  (fc): Linear(in_features=512, out_features=4, bias=True)
)

In [35]:
batch_size = 128
# 使用SGD优化器
momentum = 0.9
learning_rate = 0.1
# weight_decay = 1e-4
weight_decay = 0
optimizer = optim.SGD(model.parameters(), lr=learning_rate, momentum=momentum, weight_decay=weight_decay)
criterion = nn.CrossEntropyLoss()


# 定义测试函数
def test_acc(global_model, test_loader):
    global_model.to(device)
    global_model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = global_model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return correct / total

# 训练模型
N_EPOCHS = 200
for epoch in range(N_EPOCHS):
    train_loss = 0
    model.train()
    for texts, labels in data_loader:
        optimizer.zero_grad()
        labels = labels.to(device)
        texts = texts.to(device)
        predictions = model(texts)
        loss = criterion(predictions, labels)
        train_loss += loss.item()
        loss.backward()
        optimizer.step()

    print(f'Epoch: {epoch+1:02}')
    print(f'\tTrain Loss: {train_loss:.3f}')
    print(f'\tTest Accuracy: {test_acc(model, test_acc_loader):.3f}')
    print(f'\tTest ASR: {test_acc(model, test_asr_loader):.3f}')

    

print("Training complete")

Epoch: 01
	Train Loss: 64.706
	Test Accuracy: 0.353
	Test ASR: 0.535
Epoch: 02
	Train Loss: 57.859
	Test Accuracy: 0.401
	Test ASR: 0.065
Epoch: 03
	Train Loss: 49.462
	Test Accuracy: 0.483
	Test ASR: 0.174
Epoch: 04
	Train Loss: 39.477
	Test Accuracy: 0.577
	Test ASR: 0.143
Epoch: 05
	Train Loss: 31.148
	Test Accuracy: 0.582
	Test ASR: 0.130
Epoch: 06
	Train Loss: 23.643
	Test Accuracy: 0.651
	Test ASR: 0.038
Epoch: 07
	Train Loss: 15.280
	Test Accuracy: 0.666
	Test ASR: 0.024
Epoch: 08
	Train Loss: 11.444
	Test Accuracy: 0.671
	Test ASR: 0.034
Epoch: 09
	Train Loss: 7.624
	Test Accuracy: 0.699
	Test ASR: 0.122
Epoch: 10
	Train Loss: 5.153
	Test Accuracy: 0.685
	Test ASR: 0.080
Epoch: 11
	Train Loss: 4.184
	Test Accuracy: 0.710
	Test ASR: 0.081
Epoch: 12
	Train Loss: 2.647
	Test Accuracy: 0.695
	Test ASR: 0.157
Epoch: 13
	Train Loss: 1.760
	Test Accuracy: 0.698
	Test ASR: 0.175
Epoch: 14
	Train Loss: 1.445
	Test Accuracy: 0.705
	Test ASR: 0.197
Epoch: 15
	Train Loss: 1.220
	Test Accur

In [36]:
import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.backends.cudnn.version())

2.0.1+cu117
11.7
8500


In [37]:
device

device(type='cuda', index=1)